In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# =============================================================================
# STEP 1: UNDERSTANDING THE DATASET
# =============================================================================

import pandas as pd

# UCI Chronic Kidney Disease dataset (Rubini, Soundarapandian & Eswaran, 2015,
# UCI ML Repository, DOI 10.24432/C5G020). 400 patients seen at a hospital in
# Tamil Nadu, India, over ~2 months. See data/DATA_SOURCES.md.
df = pd.read_csv("../../data/raw/ckd.csv")   # Update path if needed

# Drop the row-id column — it's not a feature
df = df.drop(columns=["id"])


print("=" * 70)
print("DATASET SHAPE")
print("=" * 70)
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")


print("\n" + "=" * 70)
print("FEATURE NAMES")
print("=" * 70)
print(df.columns.tolist())


print("\n" + "=" * 70)
print("FIRST FIVE RECORDS")
print("=" * 70)
display(df.head())


print("\n" + "=" * 70)
print("LAST FIVE RECORDS")
print("=" * 70)
display(df.tail())


print("\n" + "=" * 70)
print("DATASET INFORMATION")
print("=" * 70)
df.info()


print("\n" + "=" * 70)
print("DATA TYPES")
print("=" * 70)
display(df.dtypes.to_frame(name="Data Type"))

print("\nNOTE: pcv, wc, and rc show up as object (text) dtype here, not")
print("numeric — even though they're numeric lab values (packed cell volume,")
print("white cell count, red cell count). This is a known quirk of this exact")
print("UCI file: stray whitespace in the raw source breaks pandas' numeric")
print("parsing. Fixed explicitly in Step 1b below rather than silently.")


print("\n" + "=" * 70)
print("MISSING VALUES OVERVIEW")
print("=" * 70)

missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum()/len(df))*100
})

display(missing_summary)


print("\n" + "=" * 70)
print("DUPLICATE RECORDS")
print("=" * 70)
print(f"Total Duplicate Rows : {df.duplicated().sum()}")


print("\n" + "=" * 70)
print("SUMMARY STATISTICS - NUMERICAL FEATURES")
print("=" * 70)
display(df.describe().T)


print("\n" + "=" * 70)
print("SUMMARY STATISTICS - CATEGORICAL FEATURES")
print("=" * 70)

categorical_cols = df.select_dtypes(include="object").columns

if len(categorical_cols) > 0:
    display(df.describe(include="object").T)
else:
    print("No categorical features found")


print("\n" + "=" * 70)
print("MEMORY USAGE")
print("=" * 70)

memory_usage = df.memory_usage(deep=True).sum() / (1024**2)

print(f"Memory Usage : {memory_usage:.2f} MB")

In [ ]:
# =============================================================================
# STEP 1b: DATA TYPE & LABEL CLEANUP (CKD-specific — not needed for the other
# two datasets, but required here before any of the later steps will work)
# =============================================================================

print("=" * 70)
print("BEFORE CLEANUP")
print("=" * 70)
print("classification unique values:", df["classification"].unique().tolist())
print("pcv dtype:", df["pcv"].dtype, "| wc dtype:", df["wc"].dtype, "| rc dtype:", df["rc"].dtype)


# 1. Strip stray whitespace/tabs from every text column (this dataset has a
#    documented "ckd\t" vs "ckd" inconsistency in the label column caused by
#    a trailing tab character in the original source file).
categorical_cols = df.select_dtypes(include="object").columns
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({"nan": np.nan, "?": np.nan})

# 2. pcv / wc / rc are genuinely numeric lab values — convert them, coercing
#    anything unparseable (leftover artifacts) to NaN rather than dropping rows.
for col in ["pcv", "wc", "rc"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 3. Encode the target as 0/1 for correlation analysis later, while keeping
#    the original text label too.
df["target"] = df["classification"].map({"ckd": 1, "notckd": 0})


print("\n" + "=" * 70)
print("AFTER CLEANUP")
print("=" * 70)
print("classification unique values:", df["classification"].unique().tolist())
print("pcv dtype:", df["pcv"].dtype, "| wc dtype:", df["wc"].dtype, "| rc dtype:", df["rc"].dtype)
print("target value counts:", df["target"].value_counts().to_dict())

In [ ]:
# =============================================================================
# STEP 2: MISSING VALUE ANALYSIS
# =============================================================================


print("=" * 70)
print("MISSING VALUE COUNT")
print("=" * 70)

missing_count = df.isnull().sum()

display(
    missing_count.to_frame(name="Missing Values")
)


print("\n" + "=" * 70)
print("MISSING VALUE PERCENTAGE")
print("=" * 70)

missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage (%)": missing_percentage.round(2)
}).sort_values("Missing Percentage (%)", ascending=False)

display(missing_summary)


print("\n" + "=" * 70)
print("FEATURES WITH MISSING VALUES")
print("=" * 70)

missing_features = missing_summary[
    missing_summary["Missing Count"] > 0
]

if len(missing_features) > 0:
    display(missing_features)
else:
    print("No missing values found in the dataset")


print("\n" + "=" * 70)
print("MISSING VALUE VISUALIZATION")
print("=" * 70)

missing_plot = missing_features

if len(missing_plot) > 0:
    plt.figure(figsize=(12,5))

    sns.barplot(
        x=missing_plot.index,
        y=missing_plot["Missing Percentage (%)"]
    )

    plt.xticks(rotation=45)
    plt.ylabel("Missing Percentage (%)")
    plt.xlabel("Features")
    plt.title("Missing Value Percentage by Feature")
    plt.tight_layout()
    plt.show()

else:
    print("No missing values to visualize")


print("\n" + "=" * 70)
print("TOTAL MISSING VALUES")
print("=" * 70)

total_missing = df.isnull().sum().sum()
total_cells = df.shape[0] * df.shape[1]

print(f"Total Missing Values : {total_missing}")
print(f"Overall Missing Rate : {total_missing/total_cells*100:.2f}%")
print()
print("This is real, substantial missingness (unlike breast cancer's 0% and")
print("diabetes\'s near-0%) — rbc alone is missing for 38% of patients.")
print("Decide and document the imputation strategy in docs/methodology.md")
print("before 02_ckd_doda.ipynb; do not default to silent mean-fill given")
print("how skewed lab values like sc (serum creatinine) are.")

In [ ]:
# =============================================================================
# STEP 3: DUPLICATE ANALYSIS
# =============================================================================


print("=" * 70)
print("TOTAL DUPLICATE RECORDS")
print("=" * 70)

duplicate_count = df.duplicated().sum()

print(f"Duplicate Rows : {duplicate_count}")


print("\n" + "=" * 70)
print("DUPLICATE PERCENTAGE")
print("=" * 70)

duplicate_percentage = (duplicate_count / len(df)) * 100

print(f"Duplicate Percentage : {duplicate_percentage:.2f}%")


print("\n" + "=" * 70)
print("DUPLICATE RECORDS PREVIEW")
print("=" * 70)

if duplicate_count > 0:
    duplicate_rows = df[df.duplicated(keep=False)]
    display(duplicate_rows.head())
else:
    print("No duplicate records found")


print("\n" + "=" * 70)
print("DATASET SIZE BEFORE DUPLICATE REMOVAL")
print("=" * 70)

print(f"Rows Before Removal : {df.shape[0]}")


print("\n" + "=" * 70)
print("DUPLICATE REMOVAL")
print("=" * 70)

df_clean = df.drop_duplicates()

print(f"Rows After Removal : {df_clean.shape[0]}")


print("\n" + "=" * 70)
print("ROWS REMOVED")
print("=" * 70)

rows_removed = df.shape[0] - df_clean.shape[0]

print(f"Total Rows Removed : {rows_removed}")

In [ ]:
print("\n" + "=" * 70)
print("DUPLICATE RECORD VISUALIZATION")
print("=" * 70)


duplicate_comparison = pd.DataFrame({
    "Dataset": ["Before Removal", "After Removal"],
    "Number of Records": [
        df.shape[0],
        df_clean.shape[0]
    ]
})


plt.figure(figsize=(8,5))

sns.barplot(
    data=duplicate_comparison,
    x="Dataset",
    y="Number of Records"
)

plt.title("Dataset Size Before and After Duplicate Removal")
plt.ylabel("Number of Records")
plt.xlabel("Dataset")
plt.show()


print("\n" + "=" * 70)
print("DUPLICATE VS UNIQUE RECORD DISTRIBUTION")
print("=" * 70)


duplicate_distribution = pd.DataFrame({
    "Record Type": [
        "Unique Records",
        "Duplicate Records"
    ],
    "Count": [
        df_clean.shape[0],
        duplicate_count
    ]
})


plt.figure(figsize=(8,5))

sns.barplot(
    data=duplicate_distribution,
    x="Record Type",
    y="Count"
)

plt.title("Unique vs Duplicate Records")
plt.ylabel("Number of Records")
plt.xlabel("Record Type")
plt.show()

In [ ]:
# =============================================================================
# STEP 4: UNIVARIATE ANALYSIS
# =============================================================================


df = df_clean.copy()


print("=" * 70)
print("NUMERICAL FEATURES")
print("=" * 70)


numerical_features = df.select_dtypes(
    include=["int64","float64"]
).columns.tolist()

numerical_features = [f for f in numerical_features if f != "target"]

print(numerical_features)



print("\n" + "=" * 70)
print("NUMERICAL FEATURE STATISTICS")
print("=" * 70)

display(df[numerical_features].describe().T)



print("\n" + "=" * 70)
print("NUMERICAL FEATURE DISTRIBUTION")
print("=" * 70)

n_features = len(numerical_features)
cols = 4
rows = (n_features + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(18, rows*3.2))
axes = axes.flatten()

for i, feature in enumerate(numerical_features):
    sns.histplot(df[feature].dropna(), kde=True, ax=axes[i], color="#4C72B0")
    axes[i].set_title(feature, fontsize=9)
    axes[i].set_xlabel("")

for j in range(i+1, len(axes)):
    axes[j].remove()

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CATEGORICAL FEATURE ANALYSIS
# =============================================================================


print("\n" + "=" * 70)
print("CATEGORICAL / BINARY FEATURES")
print("=" * 70)


categorical_features = [
    "rbc", "pc", "pcc", "ba",
    "htn", "dm", "cad", "appet", "pe", "ane"
]

# Keep only columns present in dataset
categorical_features = [
    col for col in categorical_features
    if col in df.columns
]

print(categorical_features)



print("\n" + "=" * 70)
print("CATEGORICAL FEATURE DISTRIBUTION")
print("=" * 70)


for feature in categorical_features:
    display(
        df[feature]
        .value_counts(dropna=False)
        .to_frame(name="Count")
    )

In [ ]:
# =============================================================================
# STEP 5: OUTLIER ANALYSIS
# =============================================================================


print("=" * 70)
print("NUMERICAL FEATURES FOR OUTLIER ANALYSIS")
print("=" * 70)

print(numerical_features)


print("\n" + "=" * 70)
print("OUTLIER DETECTION USING IQR METHOD")
print("=" * 70)


outlier_summary = []


for feature in numerical_features:

    Q1 = df[feature].quantile(0.25)
    Q3 = df[feature].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR


    outliers = df[
        (df[feature] < lower_limit) |
        (df[feature] > upper_limit)
    ]


    outlier_count = len(outliers)
    non_null = df[feature].notna().sum()
    outlier_percentage = (outlier_count / non_null * 100) if non_null > 0 else 0


    outlier_summary.append({
        "Feature": feature,
        "Lower Bound": round(lower_limit,2),
        "Upper Bound": round(upper_limit,2),
        "Outlier Count": outlier_count,
        "Outlier Percentage (%)": round(outlier_percentage,2)
    })


outlier_df = pd.DataFrame(outlier_summary).sort_values(
    "Outlier Percentage (%)", ascending=False
).reset_index(drop=True)

display(outlier_df)

print("\nHigh outlier rates here (e.g. su, sc, bu) are clinically expected,")
print("not necessarily data errors — CKD patients genuinely have extreme lab")
print("values compared to healthy patients. Inspect before capping/removing")
print("anything; an \'outlier\' here may be exactly the signal that matters.")

In [ ]:
print("\n" + "=" * 70)
print("BOXPLOT VISUALIZATION")
print("=" * 70)


n_features = len(numerical_features)
cols = 4
rows = (n_features + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(18, rows*3.5))
axes = axes.flatten()

for i, feature in enumerate(numerical_features):
    sns.boxplot(x=df[feature], ax=axes[i], color="#C44E52")
    axes[i].set_title(f"{feature}", fontsize=9)
    axes[i].set_xlabel("")

for j in range(i+1, len(axes)):
    axes[j].remove()

plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "=" * 70)
print("OUTLIER PERCENTAGE VISUALIZATION")
print("=" * 70)


plt.figure(figsize=(12,5))

sns.barplot(
    data=outlier_df,
    x="Feature",
    y="Outlier Percentage (%)",
    color="#C44E52"
)

plt.title("Percentage of Outliers Across Features")
plt.xlabel("Features")
plt.ylabel("Outlier Percentage (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# STEP 6: CLASS DISTRIBUTION ANALYSIS
# =============================================================================


print("=" * 70)
print("TARGET VARIABLE DISTRIBUTION")
print("=" * 70)


target_column = "target"   # 0 = notckd, 1 = ckd


print(df[target_column].value_counts())



print("\n" + "=" * 70)
print("CLASS DISTRIBUTION PERCENTAGE")
print("=" * 70)

class_distribution = pd.DataFrame({
    "Class Count": df[target_column].value_counts(),
    "Percentage (%)": (
        df[target_column]
        .value_counts(normalize=True) * 100
    ).round(2)
})


display(class_distribution)



print("\n" + "=" * 70)
print("CLASS LABEL MAPPING")
print("=" * 70)


label_names = {0: "notckd", 1: "ckd"}
for label, count in df[target_column].value_counts().items():
    print(f"Class {label} ({label_names[label]}) : {count} samples")



print("\n" + "=" * 70)
print("CLASS DISTRIBUTION VISUALIZATION")
print("=" * 70)


plt.figure(figsize=(7,5))

sns.countplot(
    data=df,
    x=target_column
)

plt.xticks([0,1], ["Not CKD (0)", "CKD (1)"])
plt.title("CKD Diagnosis Class Distribution")
plt.xlabel("Diagnosis")
plt.ylabel("Number of Patients")
plt.show()



print("\n" + "=" * 70)
print("CLASS BALANCE RATIO")
print("=" * 70)


class_ratio = (
    df[target_column]
    .value_counts()
    .min()
    /
    df[target_column]
    .value_counts()
    .max()
)


print(f"Minority/Majority Class Ratio : {class_ratio:.3f}")
print("Similar mild imbalance to breast cancer (~3:5) — use")
print("class_weight=\'balanced\' in the DODA modeling notebook as a baseline,")
print("and watch specifically for the Random Forest accuracy/recall trade-off")
print("flagged in docs/known_issues.md given how small this dataset is (n=400)")
print("compared to diabetes (n≈230K) — instability is more likely here.")

In [ ]:
# =============================================================================
# STEP 7: CORRELATION ANALYSIS
# =============================================================================


print("=" * 70)
print("CORRELATION MATRIX")
print("=" * 70)


correlation_matrix = df[numerical_features + [target_column]].corr()


display(correlation_matrix.round(2))

In [ ]:
print("\n" + "=" * 70)
print("CORRELATION HEATMAP")
print("=" * 70)


plt.figure(figsize=(12,9))


sns.heatmap(
    correlation_matrix,
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    annot=True,
    fmt=".2f",
    annot_kws={"size":7}
)


plt.title("Feature Correlation Heatmap (Numerical Features + Target)")
plt.tight_layout()
plt.show()

print("\nNote: hemoglobin, packed cell volume, specific gravity, and red")
print("cell count are strongly NEGATIVELY correlated with CKD (i.e. lower")
print("values -> more likely CKD) — consistent with CKD-associated anemia")
print("and reduced urine concentrating ability. Albumin, blood glucose,")
print("blood urea, and serum creatinine are POSITIVELY correlated, consistent")
print("with reduced kidney filtration. This is a useful cross-check for the")
print("clinical weight dictionary in config/clinical_weights/ckd.yaml.")

In [ ]:
# =============================================================================
# STEP 8: RELATIONSHIP WITH TARGET VARIABLE
# =============================================================================


print("=" * 70)
print("TARGET VARIABLE RELATIONSHIP ANALYSIS")
print("=" * 70)


print("Numerical Features:")
print(numerical_features)


print("\nCategorical/Binary Features:")
print(categorical_features)

In [ ]:
print("\n" + "=" * 70)
print("TOP 6 NUMERICAL FEATURES BY CKD STATUS")
print("=" * 70)

top_features = (
    correlation_matrix[target_column]
    .drop(target_column)
    .abs()
    .sort_values(ascending=False)
    .head(6)
    .index.tolist()
)

print("Top 6 features:", top_features)

for feature in top_features:
    print("\n")
    print("-"*60)
    print(feature)
    print("-"*60)
    display(
        df.groupby(target_column)[feature]
        .describe()
    )

In [ ]:
# =============================================================================
# NUMERICAL FEATURES VS TARGET (COMBINED VISUALIZATION)
# =============================================================================

print("=" * 70)
print("TOP FEATURES VS CKD STATUS")
print("=" * 70)


fig, axes = plt.subplots(2, 3, figsize=(16,9))
axes = axes.flatten()


for i, feature in enumerate(top_features):

    sns.boxplot(
        data=df,
        x=target_column,
        y=feature,
        ax=axes[i]
    )

    axes[i].set_xticklabels(["Not CKD", "CKD"])
    axes[i].set_title(f"{feature} vs CKD Status")
    axes[i].set_xlabel("")


plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# BINARY FEATURES VS TARGET (COMBINED VISUALIZATION)
# =============================================================================


print("=" * 70)
print("BINARY/CATEGORICAL FEATURES VS CKD STATUS")
print("=" * 70)


n_features = len(categorical_features)
cols = 4
rows = (n_features + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(18, rows*4))
axes = axes.flatten()

for i, feature in enumerate(categorical_features):

    ckd_rate = (
        df.groupby(feature)[target_column]
        .mean() * 100
    )

    sns.barplot(
        x=ckd_rate.index,
        y=ckd_rate.values,
        ax=axes[i]
    )

    axes[i].set_title(feature)
    axes[i].set_ylabel("CKD Rate (%)")
    axes[i].set_xlabel("")

for j in range(i+1, len(axes)):
    axes[j].remove()

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# STEP 9: EDA SUMMARY
# =============================================================================


print("=" * 70)
print("EXPLORATORY DATA ANALYSIS SUMMARY")
print("=" * 70)


top_pos_corr = (
    correlation_matrix[target_column].drop(target_column).sort_values(ascending=False).head(3)
)
top_neg_corr = (
    correlation_matrix[target_column].drop(target_column).sort_values().head(3)
)
top_outliers = outlier_df.head(3)

eda_summary = pd.DataFrame({
    "EDA Component": [
        "Dataset Size",
        "Features",
        "Missing Values",
        "Duplicate Records",
        "Target Variable",
        "Class Distribution",
        "Highest Positive Correlation (CKD-associated)",
        "Highest Negative Correlation (protective)",
        "Major Outlier Features"
    ],

    "Findings": [
        f"{df.shape[0]} rows and {df.shape[1]} columns",

        "14 numerical + 10 categorical predictors",

        f"{df.isnull().sum().sum()} missing values detected ({df.isnull().sum().sum()/(df.shape[0]*df.shape[1])*100:.1f}% overall - rbc worst at 38%)",

        f"{duplicate_count} duplicate rows removed",

        f"{target_column} (0 = notckd, 1 = ckd)",

        (
            f"CKD: {round((df[target_column].value_counts(normalize=True)[1])*100,2)}%, "
            f"Not CKD: {round((df[target_column].value_counts(normalize=True)[0])*100,2)}%"
        ),

        ", ".join([f"{f} ({v:.3f})" for f, v in top_pos_corr.items()]),

        ", ".join([f"{f} ({v:.3f})" for f, v in top_neg_corr.items()]),

        ", ".join([f"{row.Feature} ({getattr(row, '_5')}%)" for row in top_outliers.itertuples()])
    ]
})

display(eda_summary)

In [ ]:
# =============================================================================
# STEP 10: SAVE CLEANED DATASET
# =============================================================================


df_clean.to_csv(
    "../../data/processed/ckd_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully")

print("\nSaved File:")
print("data/processed/ckd_cleaned.csv")

print("\nDataset Shape:")
print(df_clean.shape)

print("\nNote: \'cleaned\' here refers to structural cleaning, dtype-correction and duplicate-checking —")
print("it still contains missing values (10% overall). Imputation is a")
print("modeling-stage decision, made explicitly in 02_ckd_doda.ipynb and")

print("\nNext step: 02_ckd_doda.ipynb picks up from here — imputation,")
print("feature selection, the DODA clinical re-weighting layer, and model")
print("evaluation, using the shared functions in src/doda.py.")